# Lab 07: Multi-Step GenAI Workflows

## Overview
In this lab you will orchestrate multi-step GenAI pipelines that chain multiple foundation model calls together. You will build a multi-model pipeline using direct boto3 calls, then orchestrate the same pipeline with AWS Step Functions using its native Bedrock integration. You will also explore Bedrock Flows as a visual workflow builder, implement streaming responses with the ConverseStream API, and build a semantic caching layer using Titan Embeddings to reduce latency and cost.

## Learning Objectives
- Build multi-model pipelines that chain different foundation models for different tasks
- Orchestrate GenAI workflows with AWS Step Functions and Amazon States Language (ASL)
- Understand Bedrock Flows as a managed visual workflow builder
- Implement streaming responses with the ConverseStream API
- Build a semantic caching layer using embeddings and cosine similarity

## Exam Domain
**Building Generative AI Applications (30%)** — This lab covers Task 2.4 (multi-step workflows), focusing on orchestrating multi-model pipelines with Step Functions, streaming patterns with ConverseStream, and caching strategies for production GenAI applications.

## Architecture Diagram
![Lab 07 Architecture](../assets/diagrams/lab-07-multi-step-workflows.png)

![Step Functions Pipeline](../assets/diagrams/lab-07-step-functions-pipeline.png)

In [ ]:
%pip install boto3 numpy --quiet

In [ ]:
import boto3, json, os, time
import numpy as np

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

bedrock_runtime = session.client("bedrock-runtime")
sfn_client = session.client("stepfunctions")
lambda_client = session.client("lambda")
iam = session.client("iam")
sts = session.client("sts")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
BUCKET = f"aws-genai-lab-{ACCOUNT_ID}"
BEDROCK_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/genai-lab-bedrock-role"
LAMBDA_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/genai-lab-lambda-role"

print(f"Account:      {ACCOUNT_ID}")
print(f"S3 bucket:    {BUCKET}")
print(f"Bedrock role: {BEDROCK_ROLE_ARN}")
print(f"Lambda role:  {LAMBDA_ROLE_ARN}")

## A. Multi-Model Pipeline with Direct Calls

A **multi-model pipeline** chains different FMs for different tasks based on their strengths:

| Step | Model | Why |
|------|-------|-----|
| 1. Summarize | Llama 3 8B | Fast, cheap -- good enough for straightforward summarization |
| 2. Classify | Claude Sonnet | Strong reasoning for accurate categorization |
| 3. Recommend | Claude Sonnet | Detailed, contextual recommendations |

This pattern is common in production: **cheap models for simple tasks, capable models for reasoning**.

In [ ]:
# Sample input: a technical document describing an AWS service
input_text = """Amazon SageMaker is a fully managed machine learning service. With SageMaker, 
data scientists and developers can quickly build and train machine learning models, and then 
deploy them into a production-ready hosted environment. It provides an integrated Jupyter 
authoring notebook instance for easy access to your data sources for exploration and analysis, 
so you don't have to manage servers. It also provides common machine learning algorithms that 
are optimized to run efficiently against extremely large data in a distributed environment. 
With native support for bring-your-own-algorithms and frameworks, SageMaker offers flexible 
distributed training options that adjust to your specific workflows. Deploy a model into a 
secure and scalable environment by launching it with a single click from the SageMaker console. 
Training and hosting are billed by minutes of usage, with no minimum fees and no upfront commitments."""

print(f"Input text length: {len(input_text)} characters")
print(f"Preview: {input_text[:100]}...")

### Step 1: Summarize with Llama 3 8B

Llama 3 8B is Meta's open-weight FM -- fast, low-cost, well-suited for straightforward text tasks. We use the **native InvokeModel format** (not Converse) to demonstrate provider-specific request formats.

In [ ]:
# Llama uses native InvokeModel format — different from Converse API
# Exam: know the difference between InvokeModel (provider-specific) vs Converse (unified)
response = bedrock_runtime.invoke_model(
    modelId="meta.llama3-8b-instruct-v1:0",
    contentType="application/json",
    body=json.dumps({
        "prompt": f"Summarize this text in 2 sentences: {input_text}",
        "max_gen_len": 200,
        "temperature": 0.3  # Low temp for factual summarization — less creative variation
    })
)

summary = json.loads(response["body"].read())["generation"].strip()
print("--- Step 1: Summary (Llama 3 8B) ---")
print(summary)

### Step 2: Classify with Claude

Claude excels at nuanced classification. The summary from Step 1 becomes the input -- demonstrating how one model's output feeds the next.

In [ ]:
# Converse API — unified format that works across all Bedrock models
# temperature=0 for deterministic classification (always pick the same category)
response = bedrock_runtime.converse(
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[{
        "role": "user",
        "content": [{"text": f"Classify this summary into exactly one category (compute/storage/ai-ml/database/networking/security). Reply with only the category name, nothing else.\n\nSummary: {summary}"}]
    }],
    inferenceConfig={"maxTokens": 50, "temperature": 0}
)

category = response["output"]["message"]["content"][0]["text"].strip().lower()
print("--- Step 2: Classification (Claude) ---")
print(f"Category: {category}")

### Step 3: Generate Recommendations with Claude

Final step uses **both** the summary and classification from previous steps to generate targeted recommendations.

In [ ]:
# Both previous outputs (summary + category) feed into the final prompt
# temperature=0.5 allows some creative variation in recommendations
response = bedrock_runtime.converse(
    modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[{
        "role": "user",
        "content": [{"text": f"""Based on this summary of an AWS service and its category, provide 3 specific 
best-practice recommendations for using this service effectively.

Summary: {summary}
Category: {category}

Format each recommendation as a numbered item with a brief explanation."""}]
    }],
    inferenceConfig={"maxTokens": 512, "temperature": 0.5}
)

recommendations = response["output"]["message"]["content"][0]["text"]
print("--- Step 3: Recommendations (Claude) ---")
print(recommendations)

In [ ]:
# Full pipeline summary
print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"\nInput:           {input_text[:80]}...")
print(f"\nStep 1 (Llama):  {summary[:80]}...")
print(f"\nStep 2 (Claude): {category}")
print(f"\nStep 3 (Claude): {recommendations[:120]}...")
print("\nModels used: meta.llama3-8b-instruct-v1:0, us.anthropic.claude-sonnet-4-5-20250929-v1:0")

## B. Step Functions Orchestration

Direct Python chaining works for prototypes, but production needs **error handling, retries, and observability**. **AWS Step Functions** provides this as a managed service.

Since 2023, Step Functions has a **native Bedrock integration** (`arn:aws:states:::bedrock:invokeModel`) -- no Lambda intermediary needed.

Key benefits:
- **Built-in retry/error handling** for transient API failures
- **Visual monitoring** in the AWS Console
- **Native Bedrock calls** -- eliminates Lambda overhead
- **Parallel states** for concurrent model invocations
- **Automatic I/O mapping** between steps

### Create an IAM Role for Step Functions

Step Functions needs `bedrock:InvokeModel` permission via a role with `states.amazonaws.com` trust policy.

In [ ]:
# Step Functions needs its own role — separate from the Bedrock role
SFN_ROLE_NAME = "genai-lab-sfn-bedrock-role"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "states.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

# Resource: "*" for simplicity — production should scope to specific model ARNs
bedrock_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["bedrock:InvokeModel"],
        "Resource": "*"
    }]
}

try:
    role = iam.create_role(
        RoleName=SFN_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Step Functions role for Bedrock model invocation in GenAI lab"
    )
    SFN_ROLE_ARN = role["Role"]["Arn"]
    print(f"Created role: {SFN_ROLE_ARN}")

    iam.put_role_policy(
        RoleName=SFN_ROLE_NAME,
        PolicyName="bedrock-invoke-policy",
        PolicyDocument=json.dumps(bedrock_policy)
    )
    print("Attached Bedrock invoke policy")

    # IAM propagation delay — role may not be usable immediately
    print("Waiting 10 seconds for IAM propagation...")
    time.sleep(10)
except iam.exceptions.EntityAlreadyExistsException:
    SFN_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{SFN_ROLE_NAME}"
    print(f"Role already exists: {SFN_ROLE_ARN}")

### Define the State Machine

The ASL definition replicates the three-step pipeline using native Bedrock integration.

Key ASL features:
- **`States.Format`** -- string interpolation to inject previous outputs into prompts
- **`ResultSelector`** -- extracts specific fields from the Bedrock response
- **`ResultPath`** -- stores extracted result at a JSON path, preserving prior results

In [ ]:
# Define the Step Functions state machine (Amazon States Language)
definition = {
    "Comment": "GenAI multi-step pipeline: Summarize -> Classify -> Recommend",
    "StartAt": "BuildSummarizePrompt",
    "States": {
        "BuildSummarizePrompt": {
            "Type": "Pass",
            "Parameters": {
                "prompt.$": "States.Format('Summarize this text in 2 sentences: {}', $.input_text)"
            },
            "ResultPath": "$.summarize_input",
            "Next": "Summarize"
        },
        "Summarize": {
            "Type": "Task",
            "Resource": "arn:aws:states:::bedrock:invokeModel",
            "Parameters": {
                "ModelId": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
                "ContentType": "application/json",
                "Body": {
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 200,
                    "messages": [{
                        "role": "user",
                        "content.$": "$.summarize_input.prompt"
                    }]
                }
            },
            "ResultSelector": {
                "summary.$": "$.Body.content[0].text"
            },
            "ResultPath": "$.summarize_result",
            "Next": "BuildClassifyPrompt"
        },
        "BuildClassifyPrompt": {
            "Type": "Pass",
            "Parameters": {
                "prompt.$": "States.Format('Classify this summary into exactly one category (compute/storage/ai-ml/database/networking/security). Reply with only the category name, nothing else. Summary: {}', $.summarize_result.summary)"
            },
            "ResultPath": "$.classify_input",
            "Next": "Classify"
        },
        "Classify": {
            "Type": "Task",
            "Resource": "arn:aws:states:::bedrock:invokeModel",
            "Parameters": {
                "ModelId": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
                "ContentType": "application/json",
                "Body": {
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 50,
                    "messages": [{
                        "role": "user",
                        "content.$": "$.classify_input.prompt"
                    }]
                }
            },
            "ResultSelector": {
                "category.$": "$.Body.content[0].text"
            },
            "ResultPath": "$.classify_result",
            "Next": "BuildRecommendPrompt"
        },
        "BuildRecommendPrompt": {
            "Type": "Pass",
            "Parameters": {
                "prompt.$": "States.Format('Based on this summary and category, provide 3 best-practice recommendations. Summary: {} -- Category: {} -- Format as numbered items.', $.summarize_result.summary, $.classify_result.category)"
            },
            "ResultPath": "$.recommend_input",
            "Next": "Recommend"
        },
        "Recommend": {
            "Type": "Task",
            "Resource": "arn:aws:states:::bedrock:invokeModel",
            "Parameters": {
                "ModelId": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
                "ContentType": "application/json",
                "Body": {
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 512,
                    "messages": [{
                        "role": "user",
                        "content.$": "$.recommend_input.prompt"
                    }]
                }
            },
            "ResultSelector": {
                "recommendations.$": "$.Body.content[0].text"
            },
            "ResultPath": "$.recommend_result",
            "End": True
        }
    }
}

print("State machine definition:")
print(json.dumps(definition, indent=2)[:1000] + "\n...")

### Understanding the ASL Integration

| ASL Feature | Purpose |
|-------------|---------|
| `bedrock:invokeModel` | Native Bedrock call -- no Lambda needed |
| `States.Format(...)` | Injects previous step outputs into prompts |
| `ResultSelector` | Extracts specific fields from model response |
| `ResultPath` | Stores result at a JSON path, preserving prior outputs |

Each step's output is stored at a unique path (`$.summarize_result`, `$.classify_result`, `$.recommend_result`) so context accumulates as the pipeline progresses.

### Create and Execute the State Machine

In [ ]:
# EXPRESS type = synchronous, short-lived workflows (max 5 min)
# STANDARD type = long-running, async workflows (max 1 year)
# Exam: know when to use EXPRESS vs STANDARD
SM_NAME = "genai-lab-multi-step-pipeline"

try:
    sm = sfn_client.create_state_machine(
        name=SM_NAME,
        definition=json.dumps(definition),
        roleArn=SFN_ROLE_ARN,
        type="EXPRESS"
    )
    SM_ARN = sm["stateMachineArn"]
    print(f"Created state machine: {SM_ARN}")
except sfn_client.exceptions.StateMachineAlreadyExists:
    SM_ARN = f"arn:aws:states:{REGION}:{ACCOUNT_ID}:stateMachine:{SM_NAME}"
    sfn_client.update_state_machine(
        stateMachineArn=SM_ARN,
        definition=json.dumps(definition),
        roleArn=SFN_ROLE_ARN
    )
    print(f"Updated existing state machine: {SM_ARN}")

In [ ]:
# start_sync_execution — blocks until complete (EXPRESS workflows only)
# STANDARD workflows use start_execution + polling
execution = sfn_client.start_sync_execution(
    stateMachineArn=SM_ARN,
    name=f"lab07-run-{int(time.time())}",
    input=json.dumps({"input_text": input_text})
)

print(f"Execution status: {execution['status']}")

if execution["status"] == "SUCCEEDED":
    output = json.loads(execution["output"])
    print(f"\n--- Step Functions Pipeline Results ---")
    print(f"\nSummary:  {output['summarize_result']['summary'][:200]}")
    print(f"\nCategory: {output['classify_result']['category']}")
    print(f"\nRecommendations:\n{output['recommend_result']['recommendations'][:500]}")
else:
    print(f"\nError: {execution.get('error', 'Unknown')}")
    print(f"Cause: {execution.get('cause', 'Unknown')}")

## C. Bedrock Flows (Conceptual)

**Amazon Bedrock Flows** is a visual, GenAI-specific workflow builder. Unlike Step Functions (general-purpose), Flows is purpose-built for prompt chaining.

| Criteria | Bedrock Flows | Step Functions |
|----------|---------------|----------------|
| Best for | GenAI-specific pipelines | General-purpose orchestration |
| Interface | Visual drag-and-drop | ASL JSON |
| Integrations | Bedrock models, KBs, agents | 200+ AWS services |
| Error handling | Basic retry | Advanced (Catch/Retry/Fallback) |
| Pricing | Per flow step | Per state transition |

**Flow node types:** Input, Prompt, Knowledge Base, Agent, Condition, Iterator, Collector, Lambda, S3 Storage, Output.

In [ ]:
# Bedrock Flows — API structure reference
# This shows the create_flow API structure for understanding the exam.
# Creating a fully functional flow requires additional setup (prompt templates,
# prepared agents, etc.), so we show the structure conceptually here.

bedrock_agent_client = session.client("bedrock-agent")

flow_definition = {
    "name": "genai-lab-summarize-classify",
    "description": "Summarize text then classify it using two different models",
    "executionRoleArn": BEDROCK_ROLE_ARN,
    "definition": {
        "nodes": [
            {
                "name": "FlowInput",
                "type": "Input",
                "outputs": [{"name": "document", "type": "String"}]
            },
            {
                "name": "Summarize",
                "type": "Prompt",
                "configuration": {
                    "prompt": {
                        "sourceConfiguration": {
                            "inline": {
                                "modelId": "meta.llama3-8b-instruct-v1:0",
                                "templateType": "TEXT",
                                "templateConfiguration": {
                                    "text": {
                                        "text": "Summarize this in 2 sentences: {{input}}",
                                        "inputVariables": [{"name": "input"}]
                                    }
                                },
                                "inferenceConfiguration": {
                                    "text": {"maxTokens": 200, "temperature": 0.3}
                                }
                            }
                        }
                    }
                },
                "inputs": [{"name": "input", "type": "String", "expression": "$.data"}],
                "outputs": [{"name": "modelCompletion", "type": "String"}]
            },
            {
                "name": "Classify",
                "type": "Prompt",
                "configuration": {
                    "prompt": {
                        "sourceConfiguration": {
                            "inline": {
                                "modelId": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
                                "templateType": "TEXT",
                                "templateConfiguration": {
                                    "text": {
                                        "text": "Classify into one category (compute/storage/ai-ml/database/networking): {{summary}}",
                                        "inputVariables": [{"name": "summary"}]
                                    }
                                },
                                "inferenceConfiguration": {
                                    "text": {"maxTokens": 50, "temperature": 0}
                                }
                            }
                        }
                    }
                },
                "inputs": [{"name": "summary", "type": "String", "expression": "$.data"}],
                "outputs": [{"name": "modelCompletion", "type": "String"}]
            },
            {
                "name": "FlowOutput",
                "type": "Output",
                "inputs": [{"name": "document", "type": "String", "expression": "$.data"}]
            }
        ],
        "connections": [
            {"name": "input-to-summarize", "source": "FlowInput", "target": "Summarize",
             "type": "Data", "configuration": {"data": {"sourceOutput": "document", "targetInput": "input"}}},
            {"name": "summarize-to-classify", "source": "Summarize", "target": "Classify",
             "type": "Data", "configuration": {"data": {"sourceOutput": "modelCompletion", "targetInput": "summary"}}},
            {"name": "classify-to-output", "source": "Classify", "target": "FlowOutput",
             "type": "Data", "configuration": {"data": {"sourceOutput": "modelCompletion", "targetInput": "document"}}}
        ]
    }
}

print("Bedrock Flow definition structure (conceptual):")
print(f"  Nodes: {[n['name'] for n in flow_definition['definition']['nodes']]}")
print(f"  Connections: {[c['name'] for c in flow_definition['definition']['connections']]}")
print(f"\nFlow: FlowInput -> Summarize (Llama 3) -> Classify (Claude) -> FlowOutput")
print("\nNote: To create this flow in practice, use the Bedrock console's visual builder")
print("or call bedrock_agent_client.create_flow(**flow_definition)")

## D. Streaming Responses with ConverseStream

**ConverseStream** streams tokens as they're generated instead of waiting for the full response. Critical for production:
- **~200ms to first token** vs seconds for complete response
- **Better UX** -- streaming text feels responsive
- **Memory efficient** -- process tokens as they arrive

Same unified API format as Converse (works across all Bedrock models), but returns an event stream:

| Event | Content |
|-------|---------|
| `contentBlockDelta` | Incremental text chunks |
| `messageStop` | End of response |
| `metadata` | Token usage statistics |

In [ ]:
def stream_conversation(messages, system=None):
    """Stream a conversation using ConverseStream API."""
    kwargs = {
        "modelId": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        "messages": messages,
        "inferenceConfig": {"maxTokens": 512}
    }
    if system:
        kwargs["system"] = [{"text": system}]

    # converse_stream returns an event stream instead of a complete response
    response = bedrock_runtime.converse_stream(**kwargs)

    full_response = ""
    input_tokens = 0
    output_tokens = 0

    for event in response["stream"]:
        # contentBlockDelta = incremental text — print immediately for real-time display
        if "contentBlockDelta" in event:
            chunk = event["contentBlockDelta"]["delta"]["text"]
            print(chunk, end="", flush=True)
            full_response += chunk
        # metadata arrives AFTER the response — contains token counts for cost tracking
        elif "metadata" in event:
            usage = event["metadata"].get("usage", {})
            input_tokens = usage.get("inputTokens", 0)
            output_tokens = usage.get("outputTokens", 0)

    print()
    print(f"\n[Tokens \u2014 input: {input_tokens}, output: {output_tokens}]")
    return full_response

In [ ]:
# Single streaming request
print("--- Streaming Response ---\n")
response_text = stream_conversation(
    messages=[{
        "role": "user",
        "content": [{"text": "Explain the difference between Step Functions Standard and Express workflows in 3 sentences."}]
    }]
)

### Multi-Turn Streaming Conversation

Accumulate `messages` list across turns -- ConverseStream handles multi-turn identically to non-streaming Converse.

In [ ]:
# Multi-turn streaming conversation
system_prompt = "You are an AWS solutions architect. Give concise, technical answers."
messages = []

# Turn 1
print("User: What is Amazon Bedrock?\n")
messages.append({"role": "user", "content": [{"text": "What is Amazon Bedrock?"}]})
response_1 = stream_conversation(messages, system=system_prompt)
messages.append({"role": "assistant", "content": [{"text": response_1}]})

print("\n" + "=" * 60 + "\n")

# Turn 2 — follows up on the previous answer
print("User: How does it compare to using SageMaker for inference?\n")
messages.append({"role": "user", "content": [{"text": "How does it compare to using SageMaker for inference?"}]})
response_2 = stream_conversation(messages, system=system_prompt)
messages.append({"role": "assistant", "content": [{"text": response_2}]})

print(f"\nConversation history: {len(messages)} messages")

## E. Semantic Caching

**Semantic caching** indexes responses by query meaning (not exact text). "What is S3?" and "Explain AWS S3 to me" have different text but near-identical meaning -- semantic cache serves one from the other.

Benefits:
- **Cost reduction** -- skip redundant model calls
- **Latency** -- sub-millisecond cache lookup vs seconds for FM
- **Consistency** -- equivalent queries get identical responses

**Key design decision:** similarity threshold. `0.95+` = conservative (few false hits). `0.85-0.90` = aggressive (more hits, risk of wrong answers).

In [ ]:
def get_embedding(text, max_retries=5):
    """Get embedding from Titan Embeddings V2 with retry on throttle."""
    for attempt in range(max_retries):
        try:
            response = bedrock_runtime.invoke_model(
                modelId="amazon.titan-embed-text-v2:0",
                contentType="application/json",
                body=json.dumps({"inputText": text})
            )
            return json.loads(response["body"].read())["embedding"]
        except bedrock_runtime.exceptions.ThrottlingException:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Throttled, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise


class SemanticCache:
    """Embedding-based cache using cosine similarity.
    
    Production alternative: use Redis with vector search or OpenSearch k-NN
    for persistent, distributed caching. This in-memory version is for demonstration.
    """

    def __init__(self, threshold=0.92):
        self.cache = []  # list of (embedding, query, response)
        # 0.92 threshold balances hit rate vs accuracy — tune per use case
        self.threshold = threshold
        self.hits = 0
        self.misses = 0

    def get(self, query_embedding):
        """Check cache for a semantically similar query."""
        best_sim = 0.0
        best_response = None
        best_query = None

        for emb, query, response in self.cache:
            sim = np.dot(query_embedding, emb) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(emb)
            )
            if sim > best_sim:
                best_sim = sim
                best_response = response
                best_query = query

        if best_sim >= self.threshold:
            self.hits += 1
            return best_response, best_sim, best_query
        self.misses += 1
        return None, best_sim, best_query

    def put(self, query_embedding, query, response):
        """Store a query-response pair in the cache."""
        self.cache.append((query_embedding, query, response))

    def stats(self):
        """Return cache statistics."""
        total = self.hits + self.misses
        rate = self.hits / total if total > 0 else 0
        return {"total": total, "hits": self.hits, "misses": self.misses, "hit_rate": f"{rate:.0%}"}


# Initialize cache with 0.92 threshold
cache = SemanticCache(threshold=0.92)
print("Semantic cache initialized (threshold=0.92)")

In [ ]:
def cached_query(query):
    """Query with semantic caching -- check cache first, call model on miss."""
    start = time.time()

    # Step 1: Embed the query (fast — ~50ms)
    query_embedding = get_embedding(query)

    # Step 2: Check cache — cosine similarity against all cached query embeddings
    cached_response, similarity, matched_query = cache.get(query_embedding)

    if cached_response is not None:
        elapsed = (time.time() - start) * 1000
        print(f"CACHE HIT (similarity: {similarity:.4f}, time: {elapsed:.0f}ms)")
        print(f"  Matched query: '{matched_query}'")
        print(f"  Response: {cached_response[:200]}...")
        return cached_response

    # Step 3: Cache miss — call the model (slow — ~2-5s)
    response = bedrock_runtime.converse(
        modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        messages=[{"role": "user", "content": [{"text": query}]}],
        inferenceConfig={"maxTokens": 256}
    )
    answer = response["output"]["message"]["content"][0]["text"]

    # Step 4: Store in cache for future similar queries
    cache.put(query_embedding, query, answer)

    elapsed = (time.time() - start) * 1000
    print(f"CACHE MISS (best similarity: {similarity:.4f}, time: {elapsed:.0f}ms)")
    print(f"  Response: {answer[:200]}...")
    return answer

In [ ]:
# Query 1: First query — will be a cache miss (cache is empty)
print("Query 1: 'What is Amazon S3?'")
print("-" * 40)
_ = cached_query("What is Amazon S3?")

In [ ]:
# Query 2: Semantically similar — should be a cache hit
print("Query 2: 'Explain AWS S3 to me'")
print("-" * 40)
_ = cached_query("Explain AWS S3 to me")

In [ ]:
# Query 3: Different topic — should be a cache miss
print("Query 3: 'What is AWS Lambda?'")
print("-" * 40)
_ = cached_query("What is AWS Lambda?")

In [ ]:
# Query 4: Another S3 rephrasing — should be a cache hit
print("Query 4: 'Tell me about Simple Storage Service'")
print("-" * 40)
_ = cached_query("Tell me about Simple Storage Service")

print("\n" + "=" * 40)
print("Cache Statistics:", cache.stats())

## F. Appendix -- LangChain Chains (Optional)

> **Optional.** LangChain Expression Language (LCEL) expresses the same multi-step pipeline declaratively using the `|` (pipe) operator. Useful for rapid prototyping but adds a framework dependency.

In [ ]:
%pip install langchain-aws langchain --quiet

In [ ]:
# Optional: Multi-step pipeline using LangChain LCEL
from langchain_aws import ChatBedrock
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

claude = ChatBedrock(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region_name=REGION
)

# Chain 1: Summarize
summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize this text in 2 sentences:\n\n{text}"
)
summarize_chain = summarize_prompt | claude | StrOutputParser()

# Chain 2: Classify (takes the summary as input)
classify_prompt = ChatPromptTemplate.from_template(
    "Classify this summary into one category (compute/storage/ai-ml/database/networking/security). "
    "Reply with only the category name.\n\nSummary: {summary}"
)
classify_chain = classify_prompt | claude | StrOutputParser()

# Chain 3: Recommend (takes summary and category)
recommend_prompt = ChatPromptTemplate.from_template(
    "Based on this summary and category, provide 3 best-practice recommendations.\n\n"
    "Summary: {summary}\nCategory: {category}\n\nFormat as numbered items."
)
recommend_chain = recommend_prompt | claude | StrOutputParser()

# Run the pipeline step by step using LCEL
summary_result = summarize_chain.invoke({"text": input_text})
print(f"Summary: {summary_result}\n")

category_result = classify_chain.invoke({"summary": summary_result})
print(f"Category: {category_result}\n")

recommendations_result = recommend_chain.invoke({
    "summary": summary_result,
    "category": category_result
})
print(f"Recommendations:\n{recommendations_result}")

## Cleanup

Delete the Step Functions state machine and IAM role created in this lab. No Bedrock Flows were actually created (Section C was conceptual).

In [ ]:
# Delete Step Functions state machine
try:
    sfn_client.delete_state_machine(stateMachineArn=SM_ARN)
    print(f"Deleted state machine: {SM_NAME}")
except Exception as e:
    print(f"Could not delete state machine: {e}")

# Delete IAM role (must remove inline policies first)
try:
    iam.delete_role_policy(RoleName=SFN_ROLE_NAME, PolicyName="bedrock-invoke-policy")
    iam.delete_role(RoleName=SFN_ROLE_NAME)
    print(f"Deleted IAM role: {SFN_ROLE_NAME}")
except Exception as e:
    print(f"Could not delete IAM role: {e}")

print("\nCleanup complete.")

## Key Takeaways

1. **Multi-model pipelines assign the right model to each task** — use fast, cheap models (Titan) for simple tasks like summarization and more capable models (Claude) for reasoning tasks like classification and recommendation, optimizing both cost and quality
2. **Step Functions provides production-grade GenAI orchestration** — its native Bedrock integration (`bedrock:invokeModel`) eliminates Lambda intermediaries, while ASL features like `States.Format`, `ResultSelector`, and `ResultPath` handle data flow between steps
3. **Bedrock Flows offers a visual, GenAI-specific alternative** — purpose-built for prompt chaining with drag-and-drop nodes for prompts, knowledge bases, and agents, trading Step Functions' breadth for GenAI-focused simplicity
4. **ConverseStream reduces perceived latency dramatically** — streaming the first token in ~200ms instead of waiting seconds for a complete response transforms the user experience, and the unified Converse API format works identically across all Bedrock models
5. **Semantic caching cuts cost and latency for repeated queries** — by embedding queries and comparing cosine similarity against cached responses, you can avoid redundant model invocations for semantically equivalent questions

## Key Concepts

| Concept | Definition |
|---------|-----------|
| Multi-Model Pipeline | A workflow that chains multiple foundation model calls, where each model handles a specific subtask (e.g., summarize, classify, generate) based on its strengths |
| Step Functions | AWS managed orchestration service that coordinates multi-step workflows using Amazon States Language (ASL), with native integrations for 200+ AWS services including Bedrock |
| Amazon States Language (ASL) | JSON-based language for defining Step Functions state machines, supporting task states, parallel execution, choice branching, and data transformation between steps |
| Bedrock Flows | A visual, GenAI-specific workflow builder in Amazon Bedrock that chains prompts, knowledge bases, agents, and custom logic using a drag-and-drop interface |
| ConverseStream | Bedrock streaming API that returns model responses token-by-token via an event stream, reducing perceived latency and enabling real-time text display |
| Semantic Cache | A caching layer that uses embedding similarity (cosine distance) to match semantically equivalent queries, returning cached responses without invoking the model |
| Orchestration | The coordination of multiple steps, services, or model calls in a defined sequence with error handling, retry logic, and data flow management |

## Exam Preparation — Common Exam Question Patterns

**Q: How does Step Functions integrate with Amazon Bedrock for multi-step GenAI workflows?**
A: Step Functions has a native Bedrock integration using the `arn:aws:states:::bedrock:invokeModel` resource in task states. This calls foundation models directly without Lambda functions. ASL intrinsic functions like `States.Format` inject previous step outputs into prompts, while `ResultSelector` and `ResultPath` extract and accumulate results across steps. Express workflows are ideal for synchronous GenAI pipelines.

**Q: When should you use Bedrock Flows vs Step Functions for GenAI orchestration?**
A: Use Bedrock Flows when your workflow is entirely GenAI-focused (chaining prompts, knowledge bases, and agents) and you want a visual, low-code builder. Use Step Functions when you need to integrate GenAI calls with other AWS services (DynamoDB, SQS, Lambda), require advanced error handling (Catch/Retry/Fallback), or need parallel execution patterns like Map state for batch processing.

**Q: What is the difference between ConverseStream and Converse APIs in Bedrock?**
A: Both use the same unified request format that works across all Bedrock models. Converse returns the complete response in a single API call, while ConverseStream returns an event stream that delivers tokens incrementally. ConverseStream emits events like `contentBlockDelta` (text chunks), `messageStop` (end of response), and `metadata` (token usage). Use ConverseStream for user-facing applications where perceived latency matters.

**Q: How does semantic caching reduce costs in GenAI applications?**
A: Semantic caching embeds each incoming query and compares it against cached query embeddings using cosine similarity. If a cached query exceeds the similarity threshold (e.g., 0.92), the cached response is returned without invoking the foundation model. This avoids redundant model calls for semantically equivalent queries — "What is S3?" and "Explain S3 to me" would match. The trade-off is choosing the right threshold: too high misses valid cache hits, too low returns incorrect cached responses.

**Q: What are the benefits of multi-model pipelines in production GenAI applications?**
A: Multi-model pipelines assign each subtask to the most appropriate model. Fast, inexpensive models (like Titan) handle simple tasks (summarization, extraction), while more capable models (like Claude) handle complex reasoning (classification, generation). This optimizes both cost and quality. The pipeline pattern also enables independent scaling, testing, and prompt tuning for each step, and allows swapping models without changing the overall architecture.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Llama 3 8B — Section A (summarization) | 1 call, ~300 tokens | < $0.01 |
| Claude Sonnet 4.5 — Section A (classify + recommend) | 2 calls, ~1K tokens | ~$0.02 |
| Claude Haiku 4.5 — Section B (Step Functions summarization) | 1 call, ~300 tokens | < $0.01 |
| Claude Sonnet 4.5 — Section B (Step Functions classify + recommend) | 2 calls, ~1K tokens | ~$0.02 |
| Claude Sonnet 4.5 — Section D (streaming, 3 calls) | 3 calls, ~2K tokens | ~$0.05 |
| Claude Sonnet 4.5 — Section E (semantic cache, 2 misses) | 2 calls, ~1K tokens | ~$0.03 |
| Titan Embeddings V2 — Section E (4 embeddings) | 4 calls, ~200 tokens | < $0.01 |
| Claude Sonnet 4.5 — Section F (LangChain, optional, 3 calls) | 3 calls, ~1.5K tokens | ~$0.04 |
| Step Functions — Express workflow execution | 1 execution, 3 state transitions | < $0.01 |
| **Total** | | **~$0.20** |

This lab is very cost-efficient because it primarily demonstrates orchestration patterns. The dominant cost is Claude Sonnet 4.5 API calls, which are minimal given the short prompts. Step Functions Express workflows are priced per execution (not per state transition), making them extremely economical for short-lived GenAI pipelines.